# Silver - State of Data Brasil (Tech Challenge)

Camada **analítica e normalizada**: granularidade de respondente (sem agregação), integridade garantida por chaves e valores de domínio padronizados.

| Prefixo | Tipo | Conteúdo |
|---|---|---|
| `slv_mtd_*` | Master data | tabelas de descrição: um código (`cd_`) para cada valor existente (faixa de idade, gênero, UF, opção de múltipla escolha...) |
| `slv_evt_*` | Eventos | uma linha por respondente / por resposta, só com códigos (FK), flags e medidas calculadas |
| `slv_meta_constraints` | Metadados | PKs e FKs do modelo (o Parquet não impõe constraints, então elas são validadas aqui) |

**Fluxo:** Bronze (Parquet) -> padronização de valores -> MTDs -> EVTs -> validação de PK/FK.
Processamento *full refresh* (a base é pequena): os códigos das MTDs e os eventos são sempre recriados juntos, então nunca ficam inconsistentes.

In [4]:
%idle_timeout 30
%worker_type G.1X
%number_of_workers 2

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2


In [2]:
import re, unicodedata
from pyspark.sql import functions as F

BUCKET = "tech-challenge-datahackers"
BASE   = f"s3://{BUCKET}"
BRONZE = f"{BASE}/bronze"             # pastas: respondentes/, respostas_multipla/, dim_perguntas/
SILVER = f"{BASE}/silver"

resp_brz  = spark.read.parquet(f"{BRONZE}/respondentes/")
longo_brz = spark.read.parquet(f"{BRONZE}/respostas_multipla/")
dim_brz   = spark.read.parquet(f"{BRONZE}/dim_perguntas/")

# a tabela de respondentes precisa ser a "larga" (perfil), nao a longa
assert "faixa_idade" in resp_brz.columns, (
    f"{BRONZE}/respondentes/ nao tem as colunas de perfil. Colunas encontradas: {resp_brz.columns[:8]}")
print("bronze respondentes:", resp_brz.count(), "| respostas_multipla:", longo_brz.count())

bronze respondentes: 8709 | respostas_multipla: 1026336


## 1. Configuração do modelo

Cada linha de `DOMINIOS` vira **uma MTD**: `(nome da MTD, colunas da Bronze que usam esse domínio, ordenação)`.
Quando duas colunas compartilham o mesmo domínio (ex.: modelo de trabalho atual e ideal), elas usam a **mesma MTD** com duas FKs.
Ordenação: `"numerica"` = pela primeira faixa numérica do texto; uma lista = ordem explícita; `None` = alfabética.

In [3]:
OUTROS = "Outros (resposta livre)"

DOMINIOS = [
    ("faixa_idade",            ["faixa_idade"],                                   "numerica"),
    ("genero",                 ["genero"],                                        None),
    ("cor_raca_etnia",         ["cor_raca_etnia"],                                None),
    ("pcd",                    ["pcd"],                                           None),
    ("nivel_ensino",           ["nivel_de_ensino"],                               None),
    ("area_formacao",          ["area_de_formacao"],                              None),
    ("situacao_trabalho",      ["situacao_de_trabalho"],                          None),
    ("setor",                  ["setor"],                                         None),
    ("porte_empresa",          ["numero_de_funcionarios"],                        "numerica"),
    ("cargo_gestor",           ["cargo_como_gestor"],                             None),
    ("cargo",                  ["cargo_atual"],                                   None),
    ("senioridade",            ["nivel"],                                         ["Júnior", "Pleno", "Sênior", "Especialista/Staff+"]),
    ("tempo_exp_dados",        ["tempo_de_experiencia_em_dados"],                 "numerica"),
    ("tempo_exp_ti",           ["tempo_de_experiencia_em_ti"],                    "numerica"),
    ("entrevistas_6m",         ["participou_de_entrevistas_ultimos_6m"],          None),
    ("plano_mudanca_6m",       ["planos_de_mudar_de_emprego_6m"],                 None),
    ("layoff",                 ["empresa_passou_por_layoff"],                     None),
    ("modelo_trabalho",        [("modelo_de_trabalho_atual", "cd_modelo_trabalho_atual"),
                                ("modelo_de_trabalho_ideal", "cd_modelo_trabalho_ideal")], None),
    ("atitude_retorno_presencial", ["atitude_em_caso_de_retorno_presencial"],     None),
    ("tamanho_time_dados",     ["numero_de_pessoas_em_dados"],                    "numerica"),
    ("prioridade_ia_generativa", ["ai_generativa_e_llm_e_uma_prioridade"],        None),
    ("funcao_atuacao",         ["funcao_de_atuacao"],                             None),
    ("atuacao_dados",          ["atuacao_em_dados"],                              None),
    ("linguagem",              [("linguagem_mais_usada", "cd_linguagem_mais_usada"),
                                ("linguagem_preferida", "cd_linguagem_preferida")], None),
    ("cloud",                  ["cloud_preferida"],                               None),
    ("ferramenta_bi",          ["ferramenta_de_bi_preferida"],                    None),
    ("objetivo_area_dados",    ["objetivo_na_area_de_dados"],                     None),
    ("oportunidade_buscada",   ["oportunidade_buscada"],                          None),
    ("tempo_busca_oportunidade", ["tempo_em_busca_de_oportunidade"],              None),
    ("experiencia_proc_seletivo", ["experiencia_em_processos_seletivos"],         None),
    ("tecnologia_data_lake",   ["tecnologia_data_lake"],                          None),
    ("tecnologia_data_warehouse", ["tecnologia_data_warehouse"],                  None),
    ("ferramenta_qualidade_dados", ["ferramentas_de_qualidade_de_dados_dia_a_dia"], None),
    ("pais",                   ["pais_onde_mora"],                                None),
]

# colunas sim/nao -> flags no evento
FLAGS = {"vive_no_brasil": "fl_vive_no_brasil", "vive_no_estado_de_formacao": "fl_vive_no_estado_formacao",
         "atua_como_gestor": "fl_atua_como_gestor", "satisfeito_atualmente": "fl_satisfeito_atualmente",
         "possui_data_lake": "fl_possui_data_lake", "possui_data_warehouse": "fl_possui_data_warehouse"}

# correcoes de valores (grafias diferentes da mesma resposta)
ALIAS_VALORES = {
    "numero_de_funcionarios": {"de 501 a 1000": "de 501 a 1.000"},
    "pais_onde_mora":         {"Perú": "Peru"},
    "ferramenta_de_bi_preferida": {"Apache Superset": "Superset",
                                   "Databricks Business Intelligence": "Databricks"},
}

# colunas com muita resposta livre: valores com frequencia menor que o minimo viram "Outros"
RARAS = {"objetivo_na_area_de_dados": 3}

# referencia: UFs e regioes do Brasil (codigos IBGE)
REGIOES = [(1, "Norte"), (2, "Nordeste"), (3, "Sudeste"), (4, "Sul"), (5, "Centro-oeste")]
UFS = [(11,"RO","Rondônia",1),(12,"AC","Acre",1),(13,"AM","Amazonas",1),(14,"RR","Roraima",1),(15,"PA","Pará",1),
       (16,"AP","Amapá",1),(17,"TO","Tocantins",1),(21,"MA","Maranhão",2),(22,"PI","Piauí",2),(23,"CE","Ceará",2),
       (24,"RN","Rio Grande do Norte",2),(25,"PB","Paraíba",2),(26,"PE","Pernambuco",2),(27,"AL","Alagoas",2),
       (28,"SE","Sergipe",2),(29,"BA","Bahia",2),(31,"MG","Minas Gerais",3),(32,"ES","Espírito Santo",3),
       (33,"RJ","Rio de Janeiro",3),(35,"SP","São Paulo",3),(41,"PR","Paraná",4),(42,"SC","Santa Catarina",4),
       (43,"RS","Rio Grande do Sul",4),(50,"MS","Mato Grosso do Sul",5),(51,"MT","Mato Grosso",5),
       (52,"GO","Goiás",5),(53,"DF","Distrito Federal",5)]

## 2. Funções auxiliares

- `canonizar`: limpa e unifica grafias (espaços, barras `\`, maiúsculas/acentos: `argentina`/`Argentina`, `Italia`/`Itália`) e aplica `ALIAS_VALORES`.
- `registrar_pk` / `registrar_fk`: guardam as constraints do modelo, usadas na validação final e gravadas em `slv_meta_constraints`.

In [4]:
CONSTRAINTS = []     # (tabela, tipo, colunas, tabela_ref, colunas_ref)
def registrar_pk(tab, cols):            CONSTRAINTS.append((tab, "PK", cols, None, None))
def registrar_fk(tab, cols, ref, rcols): CONSTRAINTS.append((tab, "FK", cols, ref, rcols))

def gravar(df, tab, coalesce=False):
    (df.coalesce(1) if coalesce else df).write.mode("overwrite").parquet(f"{SILVER}/{tab}/")

def sem_acento(t):
    return unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()

def chave(v):
    return re.sub(r"\s+", " ", sem_acento(v).lower()).strip()

def canonizar(coluna, contagem):
    """contagem: {valor_bruto: n}. Devolve {valor_bruto: valor_padronizado}."""
    limpo = {}
    for raw in contagem:
        v = re.sub(r"\s+", " ", raw.replace("\\", "")).strip()
        if coluna == "funcao_de_atuacao":                       # '*Engenharia de Dados:* texto...' -> 'Engenharia de Dados'
            m = re.match(r"^\*?\s*([^:*]+?)\s*[:*]", v)
            v = m.group(1) if m else v
        limpo[raw] = ALIAS_VALORES.get(coluna, {}).get(v, v)
    grupos = {}
    for raw, v in limpo.items():
        grupos.setdefault(chave(v), {})
        grupos[chave(v)][v] = grupos[chave(v)].get(v, 0) + contagem[raw]
    # grafia preferida do grupo: com acento, depois a mais frequente
    pref = {k: max(vs, key=lambda x: (x != sem_acento(x), vs[x])) for k, vs in grupos.items()}
    return {raw: pref[chave(v)] for raw, v in limpo.items()}

def chave_ordinal(v):
    s = v.lower()
    m = re.search(r"\d[\d\.]*", s)
    num = int(m.group().replace(".", "").rstrip(".") or 0) if m else 0
    if s.startswith(("menos de", "não tenho", "ainda não")): return (0, num, s)
    if s.startswith(("mais de", "acima de")):                return (2, num, s)
    return (1, num, s)

def ordenar(valores, ordem):
    if ordem == "numerica":
        return sorted(valores, key=chave_ordinal)
    if isinstance(ordem, list):
        return sorted(valores, key=lambda v: (ordem.index(v) if v in ordem else len(ordem), chave(v)))
    return sorted(valores, key=lambda v: (v == OUTROS, chave(v)))

## 3. Master data (MTD)

In [5]:
def lookup(coluna_df, mapa):
    """Traduz uma coluna para codigo usando um dict {valor_bruto: codigo} (nulo continua nulo)."""
    pares = [x for kv in mapa.items() for x in (F.lit(kv[0]), F.lit(kv[1]))]
    return F.element_at(F.create_map(*pares), coluna_df)

MAPAS = {}      # coluna_bronze -> (nome_fk, tabela_mtd, pk_mtd, {valor_bruto: cd})

def construir_mtd(nome, entradas, ordem):
    tab, pk, ds = f"slv_mtd_{nome}", f"cd_{nome}", f"ds_{nome}"
    cols = []
    for e in entradas:
        col, fk = (e, f"cd_{nome}") if isinstance(e, str) else e
        if col in resp_brz.columns:
            cols.append((col, fk))
        else:
            print(f"  aviso: coluna '{col}' nao existe na bronze (MTD {tab})")
    if not cols:
        return
    contagem = {}
    por_col = {}
    for col, _ in cols:
        por_col[col] = {r[0]: r[1] for r in resp_brz.where(F.col(col).isNotNull()).groupBy(col).count().collect()}
        for v, n in por_col[col].items():
            contagem[v] = contagem.get(v, 0) + n
    bruto_para_padrao = canonizar(cols[0][0], contagem)
    # (aplica a mesma limpeza para colunas que compartilham o dominio)
    if cols[0][0] in RARAS:
        total = {}
        for raw, v in bruto_para_padrao.items():
            total[v] = total.get(v, 0) + contagem[raw]
        bruto_para_padrao = {raw: (v if total[v] >= RARAS[cols[0][0]] else OUTROS)
                             for raw, v in bruto_para_padrao.items()}
    valores = ordenar(sorted(set(bruto_para_padrao.values())), ordem)
    cd = {v: i + 1 for i, v in enumerate(valores)}
    if ordem is not None:
        rows, schema = [(cd[v], v, cd[v]) for v in valores], f"{pk} int, {ds} string, nr_ordem int"
    else:
        rows, schema = [(cd[v], v) for v in valores], f"{pk} int, {ds} string"
    gravar(spark.createDataFrame(rows, schema), tab, coalesce=True)
    registrar_pk(tab, [pk])
    for col, fk in cols:
        MAPAS[col] = (fk, tab, pk, {raw: cd[bruto_para_padrao[raw]] for raw in por_col[col]})

for nome, entradas, ordem in DOMINIOS:
    construir_mtd(nome, entradas, ordem)
print("MTDs genericas:", len({m[1] for m in MAPAS.values()}))

MTDs genericas: 34


In [6]:
# --- faixa salarial: codigo + limites da faixa (ordenada pelo valor minimo) ---
fs = (resp_brz.where(F.col("faixa_salarial").isNotNull())
      .select("faixa_salarial", "salario_min", "salario_max").distinct().orderBy("salario_min").collect())
assert len({r[0] for r in fs}) == len(fs), "faixa_salarial com limites divergentes"
rows = [(i + 1, r[0], r[1], r[2], i + 1) for i, r in enumerate(fs)]
gravar(spark.createDataFrame(rows, "cd_faixa_salarial int, ds_faixa_salarial string, vl_minimo int, vl_maximo int, nr_ordem int"),
       "slv_mtd_faixa_salarial", coalesce=True)
registrar_pk("slv_mtd_faixa_salarial", ["cd_faixa_salarial"])
MAPAS["faixa_salarial"] = ("cd_faixa_salarial", "slv_mtd_faixa_salarial", "cd_faixa_salarial", {r[1]: r[0] for r in rows})

# --- regiao e UF (tabelas de referencia) ---
gravar(spark.createDataFrame(REGIOES, "cd_regiao int, ds_regiao string"), "slv_mtd_regiao", coalesce=True)
registrar_pk("slv_mtd_regiao", ["cd_regiao"])
gravar(spark.createDataFrame(UFS, "cd_uf int, sg_uf string, ds_uf string, cd_regiao int"), "slv_mtd_uf", coalesce=True)
registrar_pk("slv_mtd_uf", ["cd_uf"])
registrar_fk("slv_mtd_uf", ["cd_regiao"], "slv_mtd_regiao", ["cd_regiao"])
SIGLA_PARA_CD = {sg: cd for cd, sg, _, _ in UFS}

# --- pesquisa (ano) ---
pes = (resp_brz.groupBy("ano_pesquisa")
       .agg(F.min("data_hora_envio").alias("i"), F.max("data_hora_envio").alias("f")).orderBy("ano_pesquisa").collect())
rows = [(int(r[0]), int(r[0]), r[1].date() if r[1] else None, r[2].date() if r[2] else None) for r in pes]
gravar(spark.createDataFrame(rows, "cd_pesquisa int, nr_ano int, dt_inicio_coleta date, dt_fim_coleta date"),
       "slv_mtd_pesquisa", coalesce=True)
registrar_pk("slv_mtd_pesquisa", ["cd_pesquisa"])

In [ ]:
# --- perguntas e opcoes de multipla escolha ---
pares = longo_brz.select("pergunta", "opcao").distinct().collect()
perguntas = sorted({p for p, _ in pares})
cd_perg = {p: i + 1 for i, p in enumerate(perguntas)}
def nr_secao(p):
    m = re.match(r"^s(\d+)_", p);  return int(m.group(1)) if m else None
rows = [(cd_perg[p], p, nr_secao(p), re.sub(r"^s\d+_", "", p).replace("_", " ")) for p in perguntas]
gravar(spark.createDataFrame(rows, "cd_pergunta int, nm_pergunta string, nr_secao int, ds_pergunta string"),
       "slv_mtd_pergunta", coalesce=True)
registrar_pk("slv_mtd_pergunta", ["cd_pergunta"])

# rotulo da opcao: o mais recente entre os anos
rot = (dim_brz.where(F.col("tipo") == "flag")
       .groupBy("pergunta", "opcao")
       .agg(F.max(F.struct("ano_pesquisa", "rotulo_original")).getItem("rotulo_original").alias("ds_opcao")))
opc = sorted(pares)
opc_df = (spark.createDataFrame([(p, o, i + 1, cd_perg[p]) for i, (p, o) in enumerate(opc)],
                                "pergunta string, opcao string, cd_opcao int, cd_pergunta int")
          .join(rot, ["pergunta", "opcao"], "left"))
opcao_final = opc_df.select("cd_opcao", "cd_pergunta", F.col("opcao").alias("nm_opcao"),
                            F.coalesce("ds_opcao", F.col("opcao")).alias("ds_opcao"))
gravar(opcao_final, "slv_mtd_opcao", coalesce=True)
registrar_pk("slv_mtd_opcao", ["cd_opcao"])
registrar_fk("slv_mtd_opcao", ["cd_pergunta"], "slv_mtd_pergunta", ["cd_pergunta"])
print("perguntas:", len(perguntas), "| opcoes:", len(opc))

## 4. Eventos (EVT)

- `slv_evt_respondente`: 1 linha por respondente. Só códigos (FK para as MTDs), flags e medidas (`vl_idade`, `vl_salario_medio`).
  A chave `sk_respondente` é um hash determinístico de (ano, id), então não muda entre execuções.
- `slv_evt_resposta_multipla`: 1 linha por (respondente, opção). `fl_selecionado` = 1/0 apenas para quem viu a pergunta.

In [7]:
def cd_uf(sigla_col, estado_col):
    s = F.upper(F.trim(F.col(sigla_col)))
    do_estado = F.regexp_extract(F.col(estado_col), r"\((\w{2})\)", 1)
    sigla = F.coalesce(s, F.when(do_estado != "", do_estado))
    return lookup(sigla, SIGLA_PARA_CD)

evt = resp_brz.select(
    F.xxhash64("ano_pesquisa", "id_respondente").alias("sk_respondente"),
    "id_respondente",
    F.col("ano_pesquisa").cast("int").alias("cd_pesquisa"),
    F.col("data_hora_envio").alias("dt_envio"),
    F.col("idade").alias("vl_idade"),
    F.col("salario_medio").alias("vl_salario_medio"),
    *[F.col(c).alias(fl) for c, fl in FLAGS.items() if c in resp_brz.columns],
    (F.col("estado_de_origem") == "Nasci/me formei fora do Brasil").alias("fl_origem_exterior"),
    cd_uf("uf_onde_mora", "estado_onde_mora").alias("cd_uf_residencia"),
    cd_uf("uf_de_origem", "estado_de_origem").alias("cd_uf_origem"),
    *[lookup(F.col(c), m[3]).alias(m[0]) for c, m in MAPAS.items()],
)
gravar(evt, "slv_evt_respondente")
registrar_pk("slv_evt_respondente", ["sk_respondente"])
registrar_fk("slv_evt_respondente", ["cd_pesquisa"], "slv_mtd_pesquisa", ["cd_pesquisa"])
registrar_fk("slv_evt_respondente", ["cd_uf_residencia"], "slv_mtd_uf", ["cd_uf"])
registrar_fk("slv_evt_respondente", ["cd_uf_origem"], "slv_mtd_uf", ["cd_uf"])
for c, (fk, tab, pk, _) in MAPAS.items():
    registrar_fk("slv_evt_respondente", [fk], tab, [pk])

In [8]:
evt_resp = (longo_brz
    .join(F.broadcast(opc_df.select("pergunta", "opcao", "cd_opcao")), ["pergunta", "opcao"], "inner")
    .select(F.xxhash64("ano_pesquisa", "id_respondente").alias("sk_respondente"),
            "cd_opcao", F.col("selecionado").cast("int").alias("fl_selecionado")))
gravar(evt_resp, "slv_evt_resposta_multipla")
registrar_pk("slv_evt_resposta_multipla", ["sk_respondente", "cd_opcao"])
registrar_fk("slv_evt_resposta_multipla", ["sk_respondente"], "slv_evt_respondente", ["sk_respondente"])
registrar_fk("slv_evt_resposta_multipla", ["cd_opcao"], "slv_mtd_opcao", ["cd_opcao"])

# registro das constraints (documentacao/governanca)
meta = [(t, tp, ",".join(c), rt, ",".join(rc) if rc else None) for t, tp, c, rt, rc in CONSTRAINTS]
gravar(spark.createDataFrame(meta, "tabela string, tipo string, colunas string, tabela_ref string, colunas_ref string"),
       "slv_meta_constraints", coalesce=True)
print("tabelas silver gravadas, constraints:", len(CONSTRAINTS))

NameError: name 'opc_df' is not defined


## 5. Validação de integridade

O Parquet não impõe PK/FK, então o notebook verifica todas as constraints registradas:
- **PK**: sem nulos e sem duplicidade.
- **FK**: nenhum valor preenchido sem correspondência na tabela referenciada (FK nula é permitida: quem não viu a pergunta não tem código).

In [9]:
def validar():
    falhas, linhas = [], []
    for tab, tipo, cols, rtab, rcols in CONSTRAINTS:
        df = spark.read.parquet(f"{SILVER}/{tab}/")
        if tipo == "PK":
            n = df.count()
            distintos = df.select(*cols).distinct().count()
            nulos = df.where(" OR ".join(f"{c} IS NULL" for c in cols)).count()
            ok = (n == distintos and nulos == 0)
            linhas.append((tab, f"PK({','.join(cols)})", f"{n} linhas, {n - distintos} duplicadas, {nulos} nulas", "OK" if ok else "FALHA"))
        else:
            ref = (spark.read.parquet(f"{SILVER}/{rtab}/")
                   .select(*[F.col(rc).alias(c) for c, rc in zip(cols, rcols)]).distinct())
            preenchidas = df.select(*cols).where(" AND ".join(f"{c} IS NOT NULL" for c in cols))
            orfas = preenchidas.join(ref, cols, "left_anti").count()
            ok = (orfas == 0)
            linhas.append((tab, f"FK({','.join(cols)}) -> {rtab}", f"{orfas} orfas", "OK" if ok else "FALHA"))
        if not ok:
            falhas.append(linhas[-1])
    for l in linhas:
        print(f"{l[3]:6} {l[0]:28} {l[1]:60} {l[2]}")
    assert not falhas, f"{len(falhas)} constraint(s) violadas"
    print(f"\n{len(linhas)} constraints verificadas, todas OK")

validar()

OK     slv_mtd_faixa_idade          PK(cd_faixa_idade)                                           9 linhas, 0 duplicadas, 0 nulas
OK     slv_mtd_genero               PK(cd_genero)                                                4 linhas, 0 duplicadas, 0 nulas
OK     slv_mtd_cor_raca_etnia       PK(cd_cor_raca_etnia)                                        7 linhas, 0 duplicadas, 0 nulas
OK     slv_mtd_pcd                  PK(cd_pcd)                                                   3 linhas, 0 duplicadas, 0 nulas
OK     slv_mtd_nivel_ensino         PK(cd_nivel_ensino)                                          7 linhas, 0 duplicadas, 0 nulas
OK     slv_mtd_area_formacao        PK(cd_area_formacao)                                         9 linhas, 0 duplicadas, 0 nulas
OK     slv_mtd_situacao_trabalho    PK(cd_situacao_trabalho)                                     13 linhas, 0 duplicadas, 0 nulas
OK     slv_mtd_setor                PK(cd_setor)                                                

## 6. Conferência rápida

Exemplo de consulta no modelo normalizado (respondente -> MTDs). O mesmo `JOIN` vai servir de base para a Gold.

In [10]:
r  = spark.read.parquet(f"{SILVER}/slv_evt_respondente/")
sn = spark.read.parquet(f"{SILVER}/slv_mtd_senioridade/")
fs = spark.read.parquet(f"{SILVER}/slv_mtd_faixa_salarial/")
(r.join(sn, "cd_senioridade").join(fs, "cd_faixa_salarial")
  .groupBy("cd_pesquisa", "ds_senioridade")
  .agg(F.count("*").alias("n"), F.round(F.avg("vl_salario_medio"), 0).alias("salario_medio"))
  .orderBy("cd_pesquisa", "ds_senioridade").show(20, False))

+-----------+-------------------+----+-------------+
|cd_pesquisa|ds_senioridade     |n   |salario_medio|
+-----------+-------------------+----+-------------+
|2024       |Júnior             |868 |4152.0       |
|2024       |Pleno              |1376|8050.0       |
|2024       |Sênior             |1572|14493.0      |
|2025       |Especialista/Staff+|349 |19685.0      |
|2025       |Júnior             |518 |4153.0       |
|2025       |Pleno              |775 |8286.0       |
|2025       |Sênior             |858 |14319.0      |
+-----------+-------------------+----+-------------+
